In [1]:
# =====================================
# CNN1D + MAMBA ONLY TIMESERIES EXPERIMENT
# =====================================

import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.run.model.run_mamba_timeseries_classifier import RunTimeSeriesCNNMamba
from src.run.preprocess.config import _project_path

In [2]:
# =====================================
# EPOCHS EXPERIMENT
# =====================================

results = []

epochs_values = [5, 10, 20, 30]

for epochs in epochs_values:

    print(f"\n=== Running experiment with epochs={epochs} ===")

    runner = RunTimeSeriesCNNMamba(
        data_root=_project_path("PaHaW/PaHaW/PaHaW_public"),
        metadata_path=_project_path(
            "PaHaW/PaHaW/PaHaW_files/corpus_PaHaW.xlsx"
        ),
        tasks=[4, 8],
        feature_names=("x", "y"),
        engineered_features=("speed", "pressure_derivative"),

        batch_size=32,

        # Iperparametro sotto test
        epochs=epochs,

        learning_rate=1e-3,
        regularization=1e-6,
        n_splits=5,

        normalize=True,
        seed=42,

        mamba_d_model = 128,
        mamba_layers = 2,
        mamba_state = 16,
        mamba_expand = 2,
        mamba_conv = 4,
        dropout = 0.2,

        device=(
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        ),

        output_dir=_project_path(
            "experiments/test_iperparametri/cnn_mamba(timeseries)/"
            f"epochs/epochs_{epochs}"
        ),

        training_cfg={
            "metrics": [
                "accuracy",
                "f1",
                "auc",
                "precision",
                "recall",
                "specificity",
            ],

            "monitor_metric": "auc",

            "early_stopping_patience": float("inf"),
            "min_delta": 0.0,

            "results_writer": "csv",
            "save_metrics": True,
        },
    )

    summary = runner.run()

    averages = summary["averages"]

    # =====================================
    # SAVE RESULTS
    # =====================================

    results.append({
        "epochs": epochs,

        "accuracy": averages["val_accuracy"],
        "f1": averages["val_f1"],
        "auc": averages["val_auc"],
        "precision": averages["val_precision"],
        "recall": averages["val_recall"],
        "specificity": averages["val_specificity"],
    })

    # =====================================
    # PRINT RESULTS
    # =====================================

    print(f"\n=== Results for epochs={epochs} ===")

    print(f"Accuracy:    {averages['val_accuracy']:.4f}")
    print(f"F1 Score:    {averages['val_f1']:.4f}")
    print(f"AUC:         {averages['val_auc']:.4f}")
    print(f"Precision:   {averages['val_precision']:.4f}")
    print(f"Recall:      {averages['val_recall']:.4f}")
    print(f"Specificity: {averages['val_specificity']:.4f}")


# =====================================
# RESULTS DATAFRAME
# =====================================

df_epochs = pd.DataFrame(results)

print("\n=== EPOCHS EXPERIMENT RESULTS ===")

display(
    df_epochs.sort_values(
        by="auc",
        ascending=False
    )
)


=== Running experiment with epochs=5 ===

FOLD 01/5
Train patients: 60
Validation patients: 15
[WARNING] Mismatch in 00075__8_1.svc: expected 1663, found 1646.
Fold 01 | Epoch 01 | train_loss=0.6759 | val_loss=0.6472 | train_accuracy=0.5417 | val_accuracy=0.6667


KeyboardInterrupt: 